# Compare configurations

Each run is auto-labelled by **only what varies between runs** (the configuration tokens it
actually differs on). Everything is saved flat under `results/<sample>/<TITLE>/<date>/` — one
folder, descriptive PNG names (`coherence_spectra_grid.png`, `g2_spectra_grid.png`,
`R_spectra_grid.png`). Plots are interactive (**ipympl**): drag to pan, scroll / box-select to
zoom, home to reset. Full-res PNGs are still written to disk.

In [ ]:
TITLE     = "Jun23_config_sweep"            # results/<date>/<TITLE>/ ...
DATA_ROOT = "data/Jun23"                    # day folder holding the per-config run folders (switch to today, e.g. data/Jun23, once acquired)
COMPARISON_VARIABLE = "Configuration"
SORT_KEY  = "config"                        # 'config' | 'time' | 'power' | 'angle' | 'name'

In [ ]:
INTERACTIVE = True
try:
    if not INTERACTIVE:
        raise ImportError
    import ipympl  # noqa: F401
    get_ipython().run_line_magic('matplotlib', 'widget')
    print('Backend: ipympl widget (interactive - drag to pan, scroll to zoom)')
except (ImportError, AttributeError):
    try:
        get_ipython().run_line_magic('matplotlib', 'inline')
    except (NameError, AttributeError):
        pass
    print('Backend: inline (static PNGs)')
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "core.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
RESULTS_DIR = ROOT / "results"
print("Project root:", ROOT)

## 1. Discover the configuration runs

In [ ]:
from src.report import discover_runs, AnalysisReport

runs = discover_runs(ROOT / DATA_ROOT, sort_key=SORT_KEY)
print(f"Found {len(runs)} run(s):")
for r in runs:
    print(f"  - {r.run_dir.name:28s} | {r.legend_tag()}")

## 2. Build the report

One `AnalysisReport` drives everything; `v = rep.visu` is the shortcut every plot below is
built from. Each figure gets **its own cell** so you can tweak `dpi`, `ylim`, `xlim`, … and
re-run just that one. `rep.save(fig, name)` displays the figure **and** writes `<name>.png`
flat into the study folder.

In [ ]:
rep = AnalysisReport(runs, title=TITLE, results_root=RESULTS_DIR,
                     comparison_variable=COMPARISON_VARIABLE, flat_output=True)
v = rep.visu

## 3. Comparison grids

One plot per cell — edit arguments (`dpi`, `ylim`, `xlim`, …) and re-run just that one.
Colour encodes the configuration (auto-labelled from what varies between runs).

### Coherence spectra grid (overlaid configurations)

In [ ]:
fig, _ = v.plot_coherence(xlim=150)
rep.save(fig, "coherence_spectra_grid", dpi=500)

### g2(0) spectra grid (overlaid configurations)

In [ ]:
fig, _ = v.plot_g2(methods=["direct"], ylim=None, tau_min=0.3, tau_max=40, step=1.0)
rep.save(fig, "g2_spectra_grid", dpi=500)

### Cauchy-Schwarz R spectra grid (overlaid configurations)

In [ ]:
fig, _ = v.plot_R(methods=["direct"], ylim=None, tau_min=0.3, tau_max=40, step=1.0)
rep.save(fig, "R_spectra_grid", dpi=500)

## 4. Quick zoom

Drag / scroll on any figure above to zoom, or re-render one pair with explicit `xlim` / `ylim`
— shown interactively, not re-saved.

In [ ]:
ref = rep.runs[0]
fig, ax = rep.visu.plot_g2(ref.get_ch("H3T"), ref.get_ch("H5T"), methods=["direct"],
                           tau_min=0.3, tau_max=20, step=0.6, ylim=(0.9, 1.4))
rep._display(fig)